# Patient Segmentation — K-Means Clustering

**What this notebook does, in one breath:** it splits the laboratory's patients into a
few simple groups (segments) so planning decisions can be made per group instead of
averaged across everyone.

We cluster on three features, one row per patient:

| Feature | Meaning |
|---|---|
| `recency_days` | how many days since this patient last ordered |
| `age` | patient age |
| `urgent_share` | share of this patient's items marked stat/emergency |

## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

DATA_FILE = Path.cwd().parent / "dataset" / "cleaned" / "item_analysis.csv"
CLUSTERING_CHART_DIR = Path.cwd().parent / "laboratory_clustering_summary"
CLUSTERING_CHART_DIR.mkdir(exist_ok=True)


def save_chart(filename: str, fig) -> None:
    """Save a chart as a 300-dpi PNG for the PDF report, then display/close it."""
    fig.tight_layout()
    fig.savefig(f"../laboratory_clustering_summary/{filename}.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


sns.set_theme(style="whitegrid", context="notebook")
CLUSTER_FEATURES = ["recency_days", "age", "urgent_share"]
print(f"Chart images will be saved to: {CLUSTERING_CHART_DIR}")

## 2. Build the patient table

K-means needs one clean numeric row per thing we want to group. Our
raw data is one row per *ordered test*, so we first squash it down to one row per
*patient* keeping three numbers each: recency, age and urgent share.

In [ ]:
item_df = pd.read_csv(DATA_FILE, encoding="utf-8-sig")
item_df["order_day"] = pd.to_datetime(item_df["order_created_at"]).dt.normalize()
item_df["is_urgent"] = (item_df["priority"] != "routine").astype(int)

cut = pd.Timestamp("2026-01-01", tz="UTC")          # Jan 2026 = one spike day only -> out
day_counts = item_df.groupby("order_day").size()
item_2025 = item_df[item_df["order_day"] < cut]
ref_day = item_2025["order_day"].max() + pd.Timedelta(days=1)   # recency measured from 31 Dec
print(f"window kept: {item_2025['order_day'].min().date()} -> {item_2025['order_day'].max().date()} "
      f"({len(day_counts[day_counts.index < cut])} days, spike days kept: "
      f"{int((day_counts[day_counts.index < cut] > 3 * day_counts[day_counts.index < cut].median()).sum())})")

pat = (
    item_2025.groupby("patient_id")
    .agg(last_day=("order_day", "max"),
         age=("age", "first"),
         urgent_items=("is_urgent", "sum"),
         items_total=("order_item_id", "count"),
         orders=("order_id", "nunique"),
         spend=("price", "sum"),
         dept_mix=("department_name", "nunique"))
)
pat["urgent_share"] = pat.pop("urgent_items") / pat["items_total"]
pat["recency_days"] = (ref_day - pat.pop("last_day")).dt.days
display(pat[CLUSTER_FEATURES].describe().round(2))

**What this shows:** ~9,800 patients, each with three plain numbers. Recency spans the
whole year (1–365 days), age 1–80, urgent share 0–1 — meaning most patients are
mostly-routine with a minority ordering urgent work. The business columns (orders,
spend, departments) ride along quietly and are only used later to *describe* the
groups — never to build them.

## 3. Scaling, and how many groups?

Age counts up to 80 while urgent share counts up to 1 — left alone,
age would decide everything and the other two features would be ignored. So we first
put all three on the same scale. Then we try group counts 2 to 9 and ask a simple
question for each: *how clearly do the groups actually sit apart?* (the "silhouette"
score — higher = cleaner). The count with the best separation wins.

**What this shows:** k = 4 wins with the clearest separation, and it is not luck —
re-running from three different random starting points gives the same winner every
time. The code chooses k automatically, so re-running stays consistent. Honest reading:
the winning score (~0.29) says the groups overlap somewhat — useful planning segments,
not crisp walls. k = 3 is tempting for simplicity, but check what it would cost below
before preferring it.

In [ ]:
scaler = StandardScaler().fit(pat[CLUSTER_FEATURES])
X_scaled = scaler.transform(pat[CLUSTER_FEATURES])

k_table = (
    pd.DataFrame({"k": range(2, 10)})
    .assign(inertia=lambda d: [KMeans(n_clusters=k, n_init=10, random_state=42)
                                .fit(X_scaled).inertia_ for k in d["k"]],
            silhouette=lambda d: [silhouette_score(X_scaled, KMeans(n_clusters=k, n_init=10,
                                               random_state=42).fit(X_scaled).labels_)
                                  for k in d["k"]])
    .set_index("k")
)
best_k = int(k_table["silhouette"].idxmax())
display(k_table.round(3))

print("Stability check - does the winner change with a different random start?")
for seed in [0, 7, 2025]:
    s = {k: silhouette_score(X_scaled, KMeans(n_clusters=k, n_init=10, random_state=seed)
                             .fit(X_scaled).labels_) for k in [3, 4, 5]}
    print(f"  seed {seed:>4}: winners {sorted(s, key=s.get, reverse=True)} "
          f"(sil: {[f'{v:.3f}' for v in sorted(s.values(), reverse=True)]})")
print(f"\nChosen k = {best_k}")

## 4. Fit k-means and meet the segments

**Plain words:** every patient is now stamped with their nearest group. Each group's
centre is a "typical member" — the table reads like four average patients. The columns
added on the right (orders, spend, departments touched) were **not** part of the
clustering; they show what each segment is *worth* so decisions stay grounded in money,
not just structure.

In [ ]:
kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit(X_scaled)
centroids = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_),
                         columns=CLUSTER_FEATURES,
                         index=range(best_k))

# name each segment from what its centre actually looks like (not hardcoded ids)
def name_segment(row) -> str:
    if row["urgent_share"] >= 0.4:
        return "Emergency-driven"
    if row["recency_days"] >= 180:
        return "Away for months"
    if row["age"] < 40:
        return "Young & recently active"
    return "Older & recently active"

names = centroids.rename_axis("cluster").apply(name_segment, axis=1).to_dict()
pat["segment"] = pd.Series(kmeans.labels_).map(names).to_numpy()

profile = (
    pat.groupby("segment")
    .agg(patients=("segment", "size"),
         recency_days=("recency_days", "mean"),
         age=("age", "mean"),
         urgent_share=("urgent_share", "mean"),
         orders=("orders", "mean"),
         avg_spend=("spend", "mean"),
         dept_mix=("dept_mix", "mean"))
    .assign(revenue_share_pct=lambda d: d["patients"] * d["avg_spend"] / pat["spend"].sum() * 100)
)
display(profile.round(2))

**Meet the four segments (the real numbers):**

- **Young & recently active** — 3,124 patients (32%): aged ~19, last order ~2.5 months
  ago, ~24% urgent items, spends ~6,387 and touches ~5 departments. Biggest earner
  (36.6% of all revenue, tied with its twin).
- **Older & recently active** — 3,138 (32%): the *twin segment* — same recency (~78
  days), spend (~6,337), urgency and breadth, but aged ~62. Same revenue share (36.5%).
- **Emergency-driven** — 1,403 (14%): aged ~42 last seen ~5 months ago, and **57% of
  their items are stat or emergency**. Lowest volume and 11% of revenue — but they are
  the ones queuing for the fast lane.
- **Away for months** — 2,111 (22%): hasn't returned for **~8.5 months**, fewest orders
  (1.5), lowest spend (~3,992). The natural win-back list.

**And what k = 3 would have cost:** it re-merges the emergency persona into the
everyone-else blend (urgency averages back down to ~0.24 in all three groups) — the
most operationally interesting segment disappears. That's why 4 is the recommendation.

## 5. The scatter view

**Plain words:** this is the only chart the clustering gets — and it is the same story
three ways. Every dot is one patient, the colour is their segment, and the big ✕ is the
group's centre: its "typical member". If a colour hugs a corner of the chart tightly,
that segment is real.

In [ ]:
order = ["Young & recently active", "Older & recently active", "Emergency-driven", "Away for months"]
# Okabe-Ito colour-blind-safe palette: strong, distinct, no accidental "alarm red"
palette = {
    "Young & recently active": "#0072B2",   # blue
    "Older & recently active": "#E69F00",   # orange
    "Emergency-driven": "#009E73",          # green
    "Away for months": "#CC79A7",            # magenta
}
# draw big groups first, the rare emergency group last so it never drowns
DRAW_ORDER = ["Away for months", "Young & recently active",
              "Older & recently active", "Emergency-driven"]

AXIS_LABELS = {
    "recency_days": "days since last order",
    "age": "age (years)",
    "urgent_share": "share of urgent items (0-1)",
}
pairs = [("age", "urgent_share"), ("age", "recency_days"), ("recency_days", "urgent_share")]
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.4))

for ax, (x_col, y_col) in zip(axes, pairs):
    for seg in DRAW_ORDER:
        sub = pat.loc[pat["segment"] == seg]
        ax.scatter(sub[x_col], sub[y_col], s=7, alpha=0.4,
                   color=palette[seg], linewidths=0)
    ax.scatter(centroids[x_col], centroids[y_col], marker="X", s=200,
               color="white", edgecolor="black", linewidth=1.6, zorder=5, label="centre")
    ax.set_xlabel(AXIS_LABELS[x_col], fontsize=10)
    ax.set_ylabel(AXIS_LABELS[y_col], fontsize=10)
    ax.set_title(f"{AXIS_LABELS[y_col]} vs {AXIS_LABELS[x_col]}", fontsize=11)
    ax.grid(alpha=0.3)

# one shared legend under the figure (from matplotlib.lines for exact order & style)
from matplotlib.lines import Line2D
handles = [Line2D([], [], marker="o", linestyle="", markersize=8,
                  label=seg, color=palette[seg]) for seg in order]
handles.append(Line2D([], [], marker="X", linestyle="", markersize=10,
                      label="centre (typical member)", color="white",
                      markeredgecolor="black", markeredgewidth=1.6))
fig.legend(handles=handles, loc="lower center", ncols=5, fontsize=9.5,
           frameon=False, bbox_to_anchor=(0.5, -0.05))

save_chart("patient_clusters_scatter", fig)

**What the scatter actually shows, plainly (read left to right):**

- **Left panel - age vs urgency:** the **green dots (Emergency-driven) form a top band
  that runs across every age** - urgent habit has nothing to do with age. Below the
  band: **blue (young actives) hugs the left edge, orange (older actives) the right
  edge** - same urgency habits, different age. The magenta (away for months) dots hide in this
  lower crowd, because their difference is about *time*, not age or urgency.
- **Middle panel - age vs recency:** this is where age does its work. **Blue sits
  bottom-left (young, recently active), orange bottom-right (older, recently active)**,
  both inside ~150 days. The **magenta block on the far right (~180-365 days) is the
  away-for-months group at every age** - "hasn't been back for months" cuts across young and old.
- **Right panel - urgency vs recency (the diagnosis panel):** here the two active
  segments **collapse into one left-hand crowd** - on-paper proof they behave
  identically. Green holds its mid-recency, high-urgency spot; magenta owns the
  far-right, low-urgency corner. One glance at this panel tells the whole story:
  **who is recently active, who went quiet, and who comes in urgent.**
- **Centres (white X)** mark each segment's typical member; the honest caveat stands:
  colours bleed at the edges (silhouette ~0.29), so segments are planning zones, not walls.

## 6. Clustering report PDF — clustering_results.pdf

Same machinery as the prediction report (`pdf_functions`): one page holding the scatter
chart and the segments explained in plain words — written so a planner can read it
without touching the notebook.

In [ ]:
from laboratory_data_analysis.pdf_functions import ReportPDF, add_text, add_chart_section

def L(s):                       # latin-1 guard helper (fpdf 1.x constraint)
    s.encode("latin-1"); return s

def bullets(items):             # one string of "- ..." lines, no backslash tricks
    return chr(10).join(f"- {b}" for b in items)

INTRO = [
    L("The laboratory's 9,776 patients were grouped into 4 segments with K-means."),
    L("Clustering features: days since last order, patient age, and the share of each "
      "patient's items booked as stat/emergency - standardised before clustering."),
    L("The chart shows the same patients three times, each panel hiding one feature "
      "(age vs urgency, age vs last visit, last visit vs urgency). Every dot is a "
      "patient; the colour is their segment; the white X is the segment's typical "
      "member."),
]

INSIGHTS = [
    L("Four segments by cleanest separation (k = 4, stable across four random "
      "restarts): Young & recently active (3,124), Older & recently active (3,138), "
      "Away for months (2,111), Emergency-driven (1,403)."),
    L("The two active segments are twins: same visiting rhythm (~2.5 months since the "
      "last order), same urgency mix and spend - only age differs (~19 vs ~62). They "
      "are 64% of patients and ~73% of the 54.5M USD revenue."),
    L("Emergency-driven (~14% of patients) books most of its tests as stat/emergency "
      "(~57% of their items) - 11% of revenue,"),
    L("Away for months (22%) last visited ~8.5 months ago, with the fewest orders and "
      "lowest spend"),
    L("The dense line at the bottom of the urgency panels is 22% of patients who never "
      "order an urgent test"),
]

PLANNING = [
    L("Plan the fast lane around the Emergency-driven group - it is their urgent work "
      "that keeps the lane busy, not the total number of patients."),
    L("Run a campaign campaign targeting the Away-for-months segment as it is large and cheap to target.")
]

pdf = ReportPDF()
pdf.add_page()
add_text(pdf, "Patient Segmentation - K-Means Clustering Results", bold=True, font_size=14)
add_text(pdf, bullets(INTRO), spacing_after=4)
add_chart_section(pdf, "../laboratory_clustering_summary/patient_clusters_scatter.png")
add_text(pdf, "Key insights", bold=True, font_size=11, spacing_after=4)
add_text(pdf, bullets(INSIGHTS), spacing_after=6)
add_text(pdf, "Actions", bold=True, font_size=11, spacing_after=4)
add_text(pdf, bullets(PLANNING), spacing_after=5)
pdf.output("../reports/clustering_results.pdf")
print("PDF written: ../reports/clustering_results.pdf (1 page: chart + plain-word insights)")